# IGANN: Interpretable Generalized Additive Neural Network

IGANN begins with a sparse linear model and stagewise fits feature-wise extreme-learning-machine corrections. Random hidden weights stay fixed; only output coefficients are solved.


## Model in one view


$$
F_T(x)=\beta_0+\sum_j\beta_jx_j+
\eta\sum_{t=1}^{T}\sum_j w_{tj}^{\mathsf T}\sigma(a_{tj}x_j).
$$

The stagewise residual fit lets a feature remain linear unless nonlinear corrections improve the objective.



IGANN starts from a sparse linear predictor and repeatedly fits weak nonlinear
feature-wise corrections to current residuals or gradients. Each correction
uses an extreme-learning-machine block: hidden weights are random and fixed,
while output weights are obtained from a regularized linear solve. The boost
rate shrinks each stage before it enters the ensemble.

This produces a useful hierarchy. Features remain linear when nonlinear stages
offer little gain, and become nonlinear only through accumulated corrections.
Early stopping controls functional complexity, while the optional sparse stage
controls how many atomic features may participate.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#2563EB", "#F97316", "#10B981", "#8B5CF6", "#EF4444"]

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = bool(globals().get("RUN_TRAINING", False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
axes[0].scatter(X["x1"], y, s=22, alpha=0.65, color=COLORS[0], edgecolor="none")
axes[0].set(title="Response across x1", xlabel="x1", ylabel="y")

group_order = ["a", "b", "c"]
group_values = [y[X["group"].to_numpy() == level] for level in group_order]
boxes = axes[1].boxplot(group_values, tick_labels=group_order, patch_artist=True)
for patch, color in zip(boxes["boxes"], COLORS, strict=False):
    patch.set_facecolor(color)
axes[1].set(title="Response by group", xlabel="group", ylabel="y")
fig.suptitle("Synthetic mixed-feature example", fontweight="bold")
plt.show()
plt.close(fig)


## Fit the model


In [ ]:
from nampy.models import IGANNClassifier, IGANNLSS, IGANNRegressor


model = IGANNRegressor(
    n_hid=10,
    n_estimators=80,
    boost_rate=0.1,
    early_stopping=12,
    solver="native",
    sparse=0,
)
model.get_params(deep=False)


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test)
    explanation = model.explain_terms(X_test, max_bins=24, center=True)
    importance = model.term_importance(X_test, center=True)
    interaction_importance = model.interaction_importance(X_test, center=True)
    display({"R2": r2, **metrics})
    display(importance)
    display(explanation.head(12))


## Evaluate and explain


In [ ]:
if RUN_TRAINING:
    observed = np.asarray(y_test).reshape(-1)
    fitted = np.asarray(predictions).reshape(-1)
    residual = observed - fitted

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)
    axes[0].scatter(observed, fitted, s=28, alpha=0.75, color=COLORS[0])
    lo = min(observed.min(), fitted.min())
    hi = max(observed.max(), fitted.max())
    axes[0].plot([lo, hi], [lo, hi], "--", color="#334155", linewidth=1.2)
    axes[0].set(title="Observed vs predicted", xlabel="Observed", ylabel="Predicted")

    axes[1].scatter(fitted, residual, s=28, alpha=0.75, color=COLORS[1])
    axes[1].axhline(0.0, color="#334155", linestyle="--", linewidth=1.2)
    axes[1].set(title="Residual pattern", xlabel="Predicted", ylabel="Residual")

    importance_plot = (
        importance.groupby("term", as_index=False)["importance"]
        .mean()
        .sort_values("importance")
    )
    axes[2].barh(importance_plot["term"], importance_plot["importance"], color=COLORS[2])
    axes[2].set(title="Mean absolute contribution", xlabel="Link-scale importance")
    fig.suptitle("Predictive fit and global explanation", fontweight="bold")
    plt.show()
    plt.close(fig)

    main_effects = explanation.loc[
        (explanation["term_type"] == "main") & (explanation["output"] == 0)
    ].copy()
    main_effects["numeric_value"] = pd.to_numeric(main_effects["value"], errors="coerce")
    main_effects = main_effects.dropna(subset=["numeric_value"])
    if not main_effects.empty:
        numeric_terms = set(main_effects["term"])
        top_term = next(
            term
            for term in importance_plot.sort_values("importance", ascending=False)["term"]
            if term in numeric_terms
        )
        curve = main_effects.loc[main_effects["term"] == top_term].sort_values("numeric_value")
        if not curve.empty:
            fig, ax = plt.subplots(figsize=(7.5, 3.8), constrained_layout=True)
            ax.plot(curve["numeric_value"], curve["contribution"], color=COLORS[3], linewidth=2.4)
            ax.scatter(curve["numeric_value"], curve["contribution"], s=20, color=COLORS[3])
            ax.axhline(0.0, color="#64748B", linewidth=1)
            ax.set(title=f"Binned explanation: {top_term}", xlabel=top_term, ylabel="Contribution")
            plt.show()
            plt.close(fig)

    term_figures = model.plot_terms(X_test, center=True, rug=True, pages=1)
    interaction_terms = interaction_importance["term"].astype(str).tolist()
    interactions_are_raw_columns = all(
        all(
            part in X_test.columns
            and pd.api.types.is_numeric_dtype(X_test[part])
            for part in term.split(":")
        )
        for term in interaction_terms
    )
    if interaction_terms and interactions_are_raw_columns:
        model.plot_interactions(X_test)


## Model-specific controls

`solver='native'` requests the upstream-style stagewise optimizer. `sparse>0` enables ABESS feature selection and requires the optional `igann-sparse` dependency.


In [ ]:
if RUN_TRAINING:
    display(model.training_history())
    display(model.selected_features_)
    display(model.basis_metadata())
    display(model.model_complexity())


## Supported task variants

Native training supports regression and binary classification. Multiclass `IGANNClassifier` and `IGANNLSS` use the fixed basis with the shared gradient engine.


In [ ]:
task_variants = {"regression": IGANNRegressor(), "classification": IGANNClassifier(), "distributional": IGANNLSS()}


## References

- [Kraus et al. (2023), Interpretable Generalized Additive Neural Networks](https://doi.org/10.1016/j.ejor.2023.06.032).
